In [3]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
data_dir = Path('/content/drive/MyDrive/IDXExchange_Training/data')

import pandas as pd
import numpy as np
from pathlib import Path
# Run in Anaconda Prompt or terminal if needed:
# pip install pandas numpy scikit-learn xgboost matplotlib joblib

# Confirming files exist
from pathlib import Path
data_dir = Path('/content/drive/MyDrive/IDXExchange_Training/data')
training_files = [
 'CRMLSSold202509.csv', 'CRMLSSold202510.csv',
 'CRMLSSold202511.csv', 'CRMLSSold202512.csv',
 'CRMLSSold202601.csv', 'CRMLSSold202602.csv',
]
test_file = 'CRMLSSold202603.csv'
for name in training_files + [test_file]:
 path = data_dir / name
 print(name, 'exists:', path.exists())

Mounted at /content/drive
CRMLSSold202509.csv exists: True
CRMLSSold202510.csv exists: True
CRMLSSold202511.csv exists: True
CRMLSSold202512.csv exists: True
CRMLSSold202601.csv exists: True
CRMLSSold202602.csv exists: True
CRMLSSold202603.csv exists: True


In [4]:
import pandas as pd
from pathlib import Path
data_dir = Path('/content/drive/MyDrive/IDXExchange_Training/data')
training_files = [
 data_dir / 'CRMLSSold202509.csv',
 data_dir / 'CRMLSSold202510.csv',
 data_dir / 'CRMLSSold202511.csv',
 data_dir / 'CRMLSSold202512.csv',
 data_dir / 'CRMLSSold202601.csv',
 data_dir / 'CRMLSSold202602.csv',
]
dfs = [pd.read_csv(f) for f in training_files]
df = pd.concat(dfs, ignore_index=True)
print('Combined training shape:', df.shape)

/tmp/ipykernel_1738/1581274328.py:12: DtypeWarning: Columns (4,74) have mixed types. Specify dtype option on import or set low_memory=False.
  dfs = [pd.read_csv(f) for f in training_files]


Combined training shape: (119913, 78)


Filter to contain only single-family residences

In [5]:
df = df[
 (df['PropertyType'] == 'Residential') &
 (df['PropertySubType'] == 'SingleFamilyResidence')
].copy()
print('After filter:', df.shape)

After filter: (59719, 78)


In [6]:
print("Types:\n", df.dtypes)
print("\nDescription:\n", df.describe())
print(df.isnull().sum().sort_values(ascending=False).head(20))

Types:
 BuyerAgentAOR                    object
ListAgentAOR                     object
Flooring                         object
ViewYN                           object
WaterfrontYN                     object
                                 ...   
HighSchoolDistrict               object
PostalCode                       object
AssociationFee                  float64
LotSizeSquareFeet               float64
MiddleOrJuniorSchoolDistrict    float64
Length: 78, dtype: object

Description:
        OriginalListPrice    ListingKey    ClosePrice      Latitude  \
count       5.959800e+04  5.971900e+04  5.971900e+04  59715.000000   
mean        1.396503e+06  1.131968e+09  1.345655e+06     34.718181   
std         1.001576e+07  1.384466e+07  9.557516e+06      1.746312   
min         0.000000e+00  4.217759e+08  1.750000e+00    -22.863239   
25%         6.299000e+05  1.119839e+09  6.150000e+05     33.764511   
50%         8.950000e+05  1.133176e+09  8.750000e+05     34.082760   
75%         1.399000e

Convert column types to numeric columns and drop invalid rows


In [7]:
numeric_cols = [
 'ClosePrice', 'BedroomsTotal', 'BathroomsTotalInteger', 'LivingArea',
 'LotSizeSquareFeet', 'YearBuilt', 'GarageSpaces', 'Stories',
 'Latitude', 'Longitude'
]
for col in numeric_cols:
 if col in df.columns:
  df[col] = pd.to_numeric(df[col], errors='coerce')

df = df[df['ClosePrice'] > 0].copy()
df = df[df['LivingArea'] > 0].copy()
df = df.dropna(subset=['ClosePrice', 'BedroomsTotal',
 'BathroomsTotalInteger', 'LivingArea', 'PostalCode']).copy()

## Deliverable

In [9]:
display(df)

print('Row count:', len(df))
print("Columns: ", df.columns.tolist())

,BuyerAgentAOR,ListAgentAOR,Flooring,ViewYN,WaterfrontYN,BasementYN,PoolPrivateYN,OriginalListPrice,ListingKey,ListAgentEmail,...,LotSizeDimensions,LotSizeArea,MainLevelBedrooms,NewConstructionYN,GarageSpaces,HighSchoolDistrict,PostalCode,AssociationFee,LotSizeSquareFeet,MiddleOrJuniorSchoolDistrict
3,SanDiego,SanDiego,NaN,True,NaN,NaN,False,3775000.0,1137354161,Sanjay.Solomon@compass.com,...,NaN,NaN,NaN,False,3.0,NaN,92037,NaN,NaN,NaN
5,TheInlandGateway,TheInlandGateway,NaN,True,NaN,NaN,True,822000.0,1137349439,annb@loislauer.com,...,NaN,14560.00,5.0,False,2.0,Redlands Unified,92373,0.0,14560.0,NaN
6,OrangeCounty,OrangeCounty,"Carpet,Vinyl,Wood",True,NaN,NaN,False,1575000.0,1137346577,Judy@JudyMcCartyRealty.com,...,NaN,3300.00,0.0,False,2.0,Saddleback Valley Unified,92630,255.0,3300.0,NaN
8,BeverlyHillsGreaterLa,BeverlyHillsGreaterLa,"Stone,Tile,Wood",True,NaN,NaN,NaN,8022333.0,1137284778,bba@theagencyre.com,...,NaN,26981.00,NaN,False,NaN,NaN,90402,NaN,26981.0,NaN
9,OrangeCounty,OrangeCounty,NaN,False,NaN,NaN,False,1525000.0,1137269984,ks@kasere.com,...,NaN,3749.00,0.0,True,2.0,Huntington Beach Union High,92708,212.0,3749.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
119879,PacificWest,PacificWest,Laminate,False,NaN,NaN,False,575000.0,1067530441,Natalia@NataliaMarquez.com,...,NaN,5663.00,3.0,False,2.0,Perris Union High,92584,330.0,5663.0,NaN
119885,Fresno,Fresno,"Carpet,Laminate,Stone,Tile",True,NaN,NaN,False,835000.0,1064720179,dpride@sti.net,...,NaN,12.51,3.0,False,4.0,Chawanakee Unified,93602,0.0,544935.6,NaN
119886,WestSanGabrielValley,WestSanGabrielValley,NaN,True,NaN,NaN,False,3960000.0,1063839253,jacobwangrealtor@gmail.com,...,NaN,21062.00,2.0,True,4.0,Walnut Valley Unified,91789,0.0,21062.0,NaN
119904,OrangeCounty,OrangeCounty,"Carpet,Tile,Wood",True,NaN,NaN,False,1900000.0,1034032870,topjessicahong@gmail.com,...,NaN,5400.00,1.0,False,3.0,Tustin Unified,92782,120.0,5400.0,NaN


Row count: 59662
Columns:  ['BuyerAgentAOR', 'ListAgentAOR', 'Flooring', 'ViewYN', 'WaterfrontYN', 'BasementYN', 'PoolPrivateYN', 'OriginalListPrice', 'ListingKey', 'ListAgentEmail', 'CloseDate', 'ClosePrice', 'ListAgentFirstName', 'ListAgentLastName', 'Latitude', 'Longitude', 'UnparsedAddress', 'PropertyType', 'LivingArea', 'ListPrice', 'DaysOnMarket', 'ListOfficeName', 'BuyerOfficeName', 'CoListOfficeName', 'ListAgentFullName', 'CoListAgentFirstName', 'CoListAgentLastName', 'BuyerAgentMlsId', 'BuyerAgentFirstName', 'BuyerAgentLastName', 'FireplacesTotal', 'AssociationFeeFrequency', 'AboveGradeFinishedArea', 'ListingKeyNumeric', 'MLSAreaMajor', 'TaxAnnualAmount', 'CountyOrParish', 'MlsStatus', 'ElementarySchool', 'AttachedGarageYN', 'ParkingTotal', 'BuilderName', 'PropertySubType', 'LotSizeAcres', 'SubdivisionName', 'BuyerOfficeAOR', 'YearBuilt', 'StreetNumberNumeric', 'ListingId', 'BathroomsTotalInteger', 'City', 'TaxYear', 'BuildingAreaTotal', 'BedroomsTotal', 'ContractStatusChangeD